# Lab 11 — PostgreSQL JDBC and Pushdown

**Tujuan:** Membaca/menulis tabel PostgreSQL melalui JDBC dan mengamati filter pushdown.

**Lingkungan:** Spark standalone (`spark-master:7077`), Jupyter, dua worker. Jalankan sel secara berurutan. Setiap notebook dapat dijalankan sendiri.

**Metode:** Prediksi → jalankan → amati `explain()`/Spark UI → ubah satu variabel → simpulkan.

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = (
    SparkSession.builder
    .appName("Spark Training - Lab 11")
    .master("spark://spark-master:7077")
    .config("spark.executor.cores", "1")
    .config("spark.executor.memory", "1g")
    .config("spark.cores.max", "4")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("Spark:", spark.version)
print("Master:", spark.sparkContext.master)

## 1. Prepare sample data in PostgreSQL

Notebook menulis tabel khusus `spark_lab11_customers` agar tidak bergantung pada `init.sql`. PostgreSQL JDBC driver di-mount pada Spark dan Jupyter.

In [ ]:
url = "jdbc:postgresql://postgres:5432/training"
props = {
    "user": "spark",
    "password": "spark",
    "driver": "org.postgresql.Driver"
}

customers = spark.range(0, 10000, 1, 4).withColumnRenamed("id", "customer_id")
customers = customers.withColumn("region", F.when(F.col("customer_id") % 3 == 0, "Jakarta").otherwise("Other"))
customers.write.mode("overwrite").jdbc(url=url, table="spark_lab11_customers", properties=props)

## 2. Read through JDBC

In [ ]:
remote = spark.read.jdbc(url=url, table="spark_lab11_customers", properties=props)
remote.show(5)
print("Input partitions:", remote.rdd.getNumPartitions())

## 3. Compare filter pushdown

In [ ]:
filtered = remote.filter("region = 'Jakarta'")
filtered.explain("formatted")
print("Jakarta rows:", filtered.count())

## 4. Explicit parallel JDBC read

Catatan: `partitionColumn` membagi pekerjaan **baca**; ini tidak berarti tabel PostgreSQL di-partition.

In [ ]:
parallel = spark.read.jdbc(
    url=url,
    table="spark_lab11_customers",
    column="customer_id",
    lowerBound=0,
    upperBound=10000,
    numPartitions=4,
    properties=props
)
print("Input partitions:", parallel.rdd.getNumPartitions())
print("Rows:", parallel.count())

**Tugas:** bandingkan jumlah task scan JDBC dalam kedua pembacaan. `lowerBound`/`upperBound` dipakai menghitung stride partisi, bukan sebagai filter global untuk seluruh tabel.

## Refleksi peserta

Tuliskan: **(1)** apa prediksi Anda, **(2)** apa yang teramati di Spark UI atau output, **(3)** mengapa Spark berperilaku demikian. Hindari menyimpulkan dari waktu eksekusi satu kali saja.